# Compare LH Ensemble to Observations

In [1]:
import os
import numpy as np
import xarray as xr
import pandas as pd

import cartopy.crs as ccrs
import cartopy.feature as cfeature

import fates_calibration_library.utils as utils
import fates_calibration_library.clm_functions as clm
import fates_calibration_library.surface_data_functions as surface
from fates_calibration_library.ilamb_functions import get_model_da
import fates_calibration_library.plotting_functions as plotting

import matplotlib.pyplot as plt
import seaborn as sns
import importlib

In [2]:
def get_obs_mean_and_sd(df, variable):
    weighted_var = df['land_area']*df[variable]
    weighted_sd = df['land_area']*np.sqrt(df[f'{variable}_var'])
    total_land = df.land_area.sum()

    weighted_var_mean = weighted_var.sum()/total_land
    weighted_sd_mean = weighted_sd.sum()/total_land

    return weighted_var_mean, weighted_sd_mean

def extract_obs(
    obs: xr.DataArray,
    var: str,
    lats: np.ndarray,
    lons: np.ndarray,
    gridcells: np.ndarray,
    landfrac,
    surdat
) -> pd.DataFrame:
    """Extracts observations for a variable and for an input lat/lon

    Args:
        obs_ds (xr.Dataset): input dataset
        var (str): variable in question
        models (list[str]): list of models
        lats (np.ndarray): list of latitudes
        lons (np.ndarray): list of longitudes

    Returns:
        pd.DataFrame: output dataframe
    """

    models = obs.model.values
    model_dfs = []
    for i, model in enumerate(models):

        # extract observations at the chosen gridcells
        dat = np.zeros(len(lats))
        for j, (lat, lon) in enumerate(zip(lats, lons)):
            nearest_index_lat = np.abs(obs["lat"] - lat).argmin()
            nearest_index_lon = np.abs(obs["lon"] - lon).argmin()
            
            # grab data at correct lat/lon
            dat[j] = obs.sel(model=model)[nearest_index_lat, nearest_index_lon]

        pct_veg = np.zeros(len(lats))
        pct_lake = np.zeros(len(lats))
        for j, (lat, lon) in enumerate(zip(lats, lons)):
            nearest_index_lat = np.abs(surdat["lat"] - lat).argmin()
            nearest_index_lon = np.abs(surdat["lon"] - lon).argmin()
            pct_veg[j] = surdat['PCT_VEG'][nearest_index_lat, nearest_index_lon]
            pct_lake[j] = surdat['PCT_LAKE'][nearest_index_lat, nearest_index_lon]

        obs_df = pd.DataFrame(
            {
                "lat": lats,
                "lon": lons,
                "gridcell": gridcells,
                "landfrac": landfrac,
                "pct_veg": pct_veg,
                'pct_lake': pct_lake,
                f"{var}": dat,
            }
        )
        obs_df['model'] = model
        model_dfs.append(obs_df)

    return pd.concat(model_dfs)

In [35]:
def evaluate_pft_grid(pft, ensemble_file, obs, land_mask_file, mesh_file,
                      land_frac_ds_file, ilamb_dat, obs_config):
    
    pft_grid = clm.get_pft_grids(land_mask_file, mesh_file, pft)
    pft_ens = clm.get_pft_ensemble(ensemble_file, pft_grid, land_frac_ds_file, surdat)

    pft_name = all_pfts[pft-1]
    pft_id = pft_ids[pft_name]
    obs_pft = obs[obs.pft == pft_name]

    subset_lake = pft != 12
    subset_landfrac = pft != 4

    if subset_landfrac:
        obs_pft = obs_pft[obs_pft.land_frac > 0.99]
        
    if subset_lake:
        obs_pft = obs_pft[obs_pft.pct_lake < 30]

    for variable in calibration_vars:
        da_annual = get_model_da(ilamb_dat, obs_config[variable]['var'], obs_config[variable]["models"])
        da_annual_mean = da_annual.sel(year=slice(2000, 2014)).mean(dim='year')

        lats = pft_ens.isel(ensemble=0).grid1d_lat.values
        lons = pft_ens.isel(ensemble=0).grid1d_lon.values
        gridcells = pft_ens.isel(ensemble=0).gridcell.values
        landfrac = pft_ens.isel(ensemble=0).land_frac
        
        dat = extract_obs(da_annual_mean, variable, lats, lons, gridcells, landfrac, surdat)

        if subset_landfrac:
            dat = dat[dat.landfrac > 0.99]
        
        if subset_lake:
            dat = dat[dat.pct_lake < 30]

        default_ens = pft_ens.isel(ensemble=0)
        default_ens['var_corrected'] = default_ens[variable]*default_ens.land_frac

        fates_df = default_ens['var_corrected'].to_dataset(name=f"FATES_{variable}").to_pandas().reset_index()
        combined = pd.merge(dat, fates_df, on='gridcell')
        
        if subset_landfrac:
            combined = combined[combined.landfrac > 0.99]
        if subset_lake:
            combined = combined[combined.pct_lake < 30]

        plt.figure(figsize=(7, 5))
        sns.histplot(data=dat.dropna(), x=variable, hue="model", multiple="dodge", kde=True);
        plt.xlabel(f"Annual {obs_config[variable]['long_name']} ({obs_config[variable]['global_units']})")
        plt.title(f"Observed {obs_config[variable]['long_name']} for {pft_id} grids");
        plt.savefig(os.path.join(fig_dir, f"{pft_id}_{variable}_obs_hist.png"))

        fig, ax = plt.subplots(figsize=(7, 7))
        models = np.unique(combined.model)
        for model in models:
            sub = combined[combined.model == model]
            ax.scatter(sub[variable], sub[f"FATES_{variable}"], label=model)
        plt.xlabel(f'ILAMB {variable}');
        plt.ylabel(f'FATES {variable}');
        plt.legend()
        plt.plot([min(combined[variable]),
                  max(combined[variable])],
                  [min(combined[variable]), 
                   max(combined[variable])], linestyle='--', c='k')
        plt.xlabel(f"ILAMB {obs_config[variable]['long_name']} ({obs_config[variable]['global_units']})")
        plt.ylabel(f"FATES {obs_config[variable]['long_name']} ({obs_config[variable]['global_units']})")
        plt.title(f"FATES vs. ILAMB {obs_config[variable]['long_name']} for {pft_id} grids");
        plt.savefig(os.path.join(fig_dir, f"{pft_id}_{variable}_gridcomp_bymodel.png"))


        fig, ax = plt.subplots(figsize=(7, 7))
        models = np.unique(combined.model)
        sc = ax.scatter(combined[variable], combined[f"FATES_{variable}"], c=combined.lat)
        plt.xlabel(f'ILAMB {variable}');
        plt.ylabel(f'FATES {variable}');
        cbar = plt.colorbar(sc, ax=ax, orientation='vertical', pad=0.05, shrink=0.7);
        cbar.set_label('latitude')  
        plt.plot([min(combined[variable]),
                  max(combined[variable])],
                  [min(combined[variable]), 
                   max(combined[variable])], linestyle='--', c='k')
        plt.xlabel(f"ILAMB {obs_config[variable]['long_name']} ({obs_config[variable]['global_units']})")
        plt.ylabel(f"FATES {obs_config[variable]['long_name']} ({obs_config[variable]['global_units']})")
        plt.title(f"FATES vs. ILAMB {obs_config[variable]['long_name']} for {pft_id} grids");
        plt.savefig(os.path.join(fig_dir, f"{pft_id}_{variable}_gridcomp_lat.png"))


        combined['var_diff'] = combined[f"FATES_{variable}"] - combined[variable]
        vmax = np.max(np.abs(combined.var_diff))
        fig, ax = plt.subplots(figsize=(13, 6), subplot_kw=dict(projection=ccrs.Robinson()))
        ax.coastlines()
        sc = ax.scatter(combined.lon, combined.lat, transform=ccrs.PlateCarree(),
                        c=combined.var_diff, cmap='RdBu_r', vmin=-1*vmax, vmax=vmax)
        cbar = plt.colorbar(sc, ax=ax, orientation='horizontal', pad=0.05, shrink=0.7)
        cbar.set_label(f"{variable} Difference ({obs_config[variable]['global_units']})", size=10, fontweight="bold")
        plt.title(f"FATES - ILAMB {obs_config[variable]['long_name']} for {pft_id} grids");
        plt.savefig(os.path.join(fig_dir, f"{pft_id}_{variable}_gridcomp_map.png"))

        pft_mean = clm.weighted_mean(pft_ens.where(pft_ens.land_frac > 0.99, drop=True), variable)
        obs_mean, obs_sd = get_obs_mean_and_sd(obs_pft[obs_pft.land_frac > 0.99], obs_config[variable]['var'])
        plotting.plot_sample(pft_mean, obs_mean, obs_sd, pft_id, variable, obs_config[variable]['global_units'])
        plt.savefig(os.path.join(fig_dir, f"{pft_id}_{variable}_ens_compare.png"))

## Set Up
Load files, set up ensemble information

In [23]:
# dataset with land area 
land_frac_ds_file = os.path.join("/glade/derecho/scratch/afoster/archive",
                            "ctsm60SP_bigleaf_fullgrid/lnd/hist",
                            "ctsm60SP_bigleaf_fullgrid.clm2.h0.0001-02-01-00000.nc")

# surface file
surdat_dir = "/glade/campaign/cesm/cesmdata/inputdata/lnd/clm2/surfdata_esmf/ctsm5.4.0/"
surdat_2deg = os.path.join(surdat_dir, "surfdata_1.9x2.5_hist_2000_16pfts_c250617.nc")

pft_id_config = '/glade/work/afoster/FATES_calibration/fates_calibration_library/configs/fates_pft_ids.yaml'
pft_ids = utils.get_config_file(pft_id_config)

# information about variables
obs_config_file = '/glade/work/afoster/FATES_calibration/fates_calibration_library/configs/ilamb_conversion.yaml'
obs_config = utils.get_config_file(obs_config_file)

# directories
mesh_dir = '/glade/work/afoster/FATES_calibration/mesh_files'
hist_dir = '/glade/work/afoster/FATES_calibration/history_files/compiled_files'
fig_dir = '/glade/work/afoster/FATES_calibration/figures/lh_figs'
param_dir = '/glade/work/afoster/FATES_calibration/parameter_files'

# grab pft names
default_param = xr.open_dataset(os.path.join(param_dir,
                                             'fates_params_default_sci.1.85.1_api.40.0.0_crops.nc'))
all_pfts = [str(pft).replace("b'", "").replace("'", "").strip() for pft in default_param.fates_pftname.values]

# variables to emulate
calibration_vars = ['GPP', 'EFLX_LH_TOT', 'FSH', 'EF']

ilamb_obs_file = '/glade/work/afoster/FATES_calibration/observations/all_ILAMB_obs.nc'
ilamb_dat = xr.open_dataset(ilamb_obs_file)

In [24]:
# information about each ensemble
ens_dict = {'dompft':
            {'mesh_file': os.path.join(mesh_dir, 'dominant_grid_mesh.nc'),
             'land_mask_file': os.path.join(mesh_dir, 'dominant_grid.nc'),
             'ensemble_file': os.path.join(hist_dir, 'fates_dompft_annual_means.nc'),
             'lhc_key_file': os.path.join(param_dir, 'fates_lh', 'fates_lh_key.csv'),
             'pfts': [1, 2, 3, 12, 13],
             'obs_df': os.path.join(mesh_dir, 'dominant_grid.csv'),
            },
            'codompft':
            {'mesh_file': os.path.join(mesh_dir, 'co-dominant_grid_mesh.nc'),
             'land_mask_file': os.path.join(mesh_dir, 'co-dominant_grid.nc'),
             'ensemble_file': os.path.join(hist_dir, 'fates_codompft_annual_means.nc'),
             'lhc_key_file': os.path.join(param_dir, 'fates_lh_codom', 'fates_lh_key.csv'),
             'pfts': [4, 6, 11, 14],
             'obs_df': os.path.join(mesh_dir, 'co-dominant_grid.csv'),
            }
           }

In [6]:
surdat = surface.get_surdat(surdat_2deg)

In [8]:
# choose ensemble
ensemble = 'codompft'

In [16]:
# read in observations
obs = pd.read_csv(ens_dict[ensemble]['obs_df'], index_col=[0])

In [ ]:
# plot some outputs
evaluate_pft_grid(14, ens_dict[ensemble]['ensemble_file'], obs,
                  ens_dict[ensemble]['land_mask_file'],
                  ens_dict[ensemble]['mesh_file'],
                  land_frac_ds_file, ilamb_dat, obs_config)